# How the PPMI tables relate

This notebook maps every table you have loaded: the keys that join them, how well those keys match, which participants and visits they share, and how the clean layer is built from the raw tables. It reads the downloads described in `dataset.ipynb`, works with whatever subset you have, and prints counts only.

PPMI's tables hang off three keys:

- `PATNO`, the participant. `Participant_Status` is the registry, with one row per participant, their cohort and their enrolment status.
- `PATNO` with `EVENT_ID`, the visit: SC (screening), BL (baseline), V01, V02 and onwards, and codes such as LOG for running logs. Assessments made at the same visit join on both.
- `IMAGEID`, the image. Derived imaging tables carry the ID of the scan they were measured on, and image searches and downloads list the same IDs.

Dates are recorded to the month, so a visit code and a month are the two ways to line measurements up in time.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import sys
sys.path.append("../code")

import pandas as pd

import data
import plot
from ppmi_pipeline import pipeline, sources

pd.set_option("display.max_rows", 300, "display.max_colwidth", 160)
p = data.PPMI()
tables = sorted(p.files) + ["images", "image_files"]
columns = {name: set(p.db.table(name).columns) for name in tables}

## 1. Keys and grain

For each raw table: its key columns, its rows, participants and participant-visits, and how many rows it has per participant-visit. A visit table has about one row per participant-visit; a log or a scan table can have more.

In [ ]:
KEYS = {"PATNO": "PATNO", "EVENT_ID": "EVENT_ID", "event": "EVENT_ID", "IMAGEID": "IMAGEID", "image_id": "IMAGEID", "REC_ID": "REC_ID", "INFODT": "INFODT"}


def grain(name):
    keys = {KEYS[column] for column in columns[name] if column in KEYS}
    visit = next((column for column in ("EVENT_ID", "event") if column in columns[name]), None)
    row = p.sql(f"""
        SELECT count(*) AS rows,
               {"count(DISTINCT PATNO)" if "PATNO" in columns[name] else "NULL"} AS participants,
               {f"count(DISTINCT PATNO || '|' || {visit})" if visit and "PATNO" in columns[name] else "NULL"} AS "participant-visits"
        FROM "{name}"
    """).iloc[0]
    return {"table": name, **{key: "yes" if key in keys else "" for key in dict.fromkeys(KEYS.values())}, **row.to_dict(), "columns": len(columns[name])}


profile = pd.DataFrame(map(grain, tables)).set_index("table")
profile["rows per visit"] = (profile.rows / profile["participant-visits"]).round(2)
profile

## 2. How the tables connect

Each table is linked to the keys it can be joined on, and two tables linked to the same key join on it. A table with a visit key also joins on `PATNO` alone, so only its visit link is drawn. Image tables carry both an image ID and a visit. Numbers in brackets are participants.

The table below the figure checks every link against a registry: `participant_status` for participants, `age_at_visit` for visits (it records age at every visit) and the image IDs in `images` and `image_files` for images. `share` is the fraction of a table's keys that the registry knows: a share well below 100% means rows drop out when you join.

In [ ]:
p.db.execute(r"""
    CREATE OR REPLACE TEMP VIEW image_index AS
    SELECT regexp_replace(trim(image_id), '^I|\.0+$', '', 'g') AS IMAGEID FROM images WHERE image_id IS NOT NULL
    UNION SELECT regexp_replace(trim(image_id), '^I|\.0+$', '', 'g') FROM image_files WHERE image_id IS NOT NULL
""")
IMAGE = r"regexp_replace(trim({}), '^I|\.0+$', '', 'g')"
REGISTRIES = {
    "participant": ("PATNO", "participant_status", ["PATNO"]),
    "visit": ("PATNO + EVENT_ID", "age_at_visit", ["PATNO", "EVENT_ID"]),
    "image": ("IMAGEID", "image_index", ["IMAGEID"]),
}
LAYERS = ["participant and visit tables", "keys", "image tables", "reference"]


def named(name, key):
    return {"EVENT_ID": "event", "IMAGEID": "image_id"}.get(key, key) if name in ("images", "image_files") else key


def link(name, kind):
    key, registry, keys = REGISTRIES[kind]
    mine = [IMAGE.format(named(name, k)) if k == "IMAGEID" else named(name, k) for k in keys]
    row = p.sql(f"""
        WITH t AS (SELECT DISTINCT {", ".join(f"{e} AS k{i}" for i, e in enumerate(mine))} FROM "{name}"),
             r AS (SELECT DISTINCT {", ".join(f"{k} AS k{i}" for i, k in enumerate(keys))} FROM {registry})
        SELECT count(*) AS keys, count(r.k0) AS matched
        FROM t LEFT JOIN r ON {" AND ".join(f"t.k{i} = r.k{i}" for i in range(len(keys)))}
        WHERE {" AND ".join(f"t.k{i} IS NOT NULL" for i in range(len(keys)))}
    """).iloc[0]
    return {"table": name, "key": key, "registry": registry, "keys": int(row["keys"]), "matched": int(row["matched"])}


kinds = {name: [kind for kind, (key, registry, keys) in REGISTRIES.items() if all(named(name, k) in columns[name] for k in keys)]
         for name in tables}
edges = pd.DataFrame([(name, REGISTRIES[kind][0], kind, REGISTRIES[kind][0]) for name in tables for kind in kinds[name]
                      if not (kind == "participant" and "visit" in kinds[name])], columns=["source", "target", "kind", "key"])
nodes = pd.concat([
    pd.DataFrame({"layer": [LAYERS[2] if "image" in kinds[name] else LAYERS[0] if kinds[name] else LAYERS[3] for name in tables],
                  "participants": profile.participants.reindex(tables)}, index=tables),
    pd.DataFrame({"layer": LAYERS[1], "participants": None}, index=[key for key, registry, keys in REGISTRIES.values()]),
]).sort_values("layer", key=lambda s: s.map(LAYERS.index), kind="stable")
plot.relations(nodes, edges, "How the downloaded tables connect")

In [ ]:
links = pd.DataFrame([link(name, kind) for name in tables for kind in kinds[name]
                      if name != REGISTRIES[kind][1] and REGISTRIES[kind][1] in set(p.files) | {"image_index"}],
                     columns=["table", "key", "registry", "keys", "matched"])
links.assign(share=(links.matched / links["keys"]).map("{:.1%}".format)).sort_values(["key", "table"])

## 3. Shared participants

How many participants each pair of tables has in common; the diagonal is each table's own count. Measures from two tables can only be analysed together for the participants they share.

In [ ]:
people = [name for name in tables if "PATNO" in columns[name]]
p.db.execute("CREATE OR REPLACE TEMP TABLE membership AS " + " UNION ALL ".join(
    f"SELECT DISTINCT '{name}' AS tbl, PATNO FROM \"{name}\" WHERE PATNO IS NOT NULL" for name in people))
shared = p.sql("SELECT a.tbl AS a, b.tbl AS b, count(*) AS n FROM membership a JOIN membership b USING (PATNO) GROUP BY ALL")
plot.heatmap(shared.pivot(index="a", columns="b", values="n").reindex(index=people, columns=people), "Participants shared by each pair of tables")

## 4. Visits each table covers

Participants at each visit code, for every table with a visit. In the original schedule V04, V06, V08, V10 and V12 fall at months 12, 24, 36, 48 and 60; later, remote and unscheduled visits have other codes. Rare codes are left out of the figure and listed below it.

In [ ]:
visit_tables = [name for name in sorted(p.files) if {"PATNO", "EVENT_ID"} <= columns[name]]
by_visit = p.sql(" UNION ALL ".join(
    f"SELECT '{name}' AS tbl, EVENT_ID, count(DISTINCT PATNO) AS n FROM \"{name}\" WHERE EVENT_ID IS NOT NULL GROUP BY ALL"
    for name in visit_tables)).pivot(index="tbl", columns="EVENT_ID", values="n")
codes = sorted(by_visit.columns, key=lambda code: ({"SC": 0, "BL": 1}.get(code, 2), code[0] != "V", len(code), code))
common = [code for code in codes if by_visit[code].max() >= 0.02 * by_visit.max().max()]
print("Left out:", ", ".join(code for code in codes if code not in common) or "nothing")
plot.heatmap(by_visit[common], "Participants at each visit, by table")

## 5. How the clean layer is built

The pipeline in `code/ppmi_pipeline` reads the raw tables and builds typed tables, then views on top of them. This figure is read from the pipeline's SQL (`schema.sql` and `views.sql`), so it always matches the code. Arrows point from a source to what is built from it, and numbers in brackets are participants.

In [ ]:
present = {sources.source_for(name): name for name in p.files if sources.source_for(name)}
built = pd.DataFrame(pipeline.lineage(), columns=["source", "target"])
built["source"] = [present.get(s[4:], f"{sources.SOURCES[s[4:]].prefixes[0]} (not downloaded)") if s.startswith("raw:") else f"clean.{s}" for s in built.source]
built["target"] = "clean." + built.target
built = built.assign(kind="built from", key="pipeline SQL")
types = dict(p.sql("SELECT 'clean.' || table_name, table_type FROM information_schema.tables WHERE table_catalog = 'clean'").itertuples(index=False))


def participants(name):
    if name.startswith("clean.") and "patient_id" in p.sql(f"SELECT * FROM {name} LIMIT 0").columns:
        return int(p.sql(f"SELECT count(DISTINCT patient_id) AS n FROM {name}").n[0])
    return int(profile.participants.get(name, 0) or 0)


names = list(dict.fromkeys([*built.source[~built.source.isin(built.target)], *built.target]))
lineage = pd.DataFrame({
    "layer": ["clean views" if types.get(name) == "VIEW" else "clean tables" if name.startswith("clean.") else "raw tables" for name in names],
    "participants": [participants(name) for name in names],
}, index=names).sort_values("layer", key=lambda s: s.map({"raw tables": 0, "clean tables": 1, "clean views": 2}), kind="stable")
plot.relations(lineage, built, "How the clean layer is built from the raw tables")

In [ ]:
p.sql("""
    SELECT 'clean.' || table_name AS "table", table_type AS kind
    FROM information_schema.tables WHERE table_catalog = 'clean' ORDER BY kind, "table"
""").set_index("table").join(built.groupby("target").source.apply(", ".join).rename("built from"))

## 6. Every column

The columns of every table in both layers. If you downloaded the Data Dictionary (Annotated), its descriptions are added to the raw columns. The dictionary names each table with a short code (`MOD_NAME`); its rows with a blank `ITM_NAME` give the file name that each code belongs to.

In [ ]:
described = p.schema()
if "data_dictionary_annotated" in p.files:
    dictionary = p.sql("""
        WITH d AS (SELECT * FROM data_dictionary_annotated),
             files AS (SELECT MOD_NAME, DSCR AS file FROM d WHERE coalesce(trim(ITM_NAME), '') = '')
        SELECT f.file, d.ITM_NAME AS "column", d.DSCR AS description
        FROM d JOIN files f USING (MOD_NAME) WHERE coalesce(trim(d.ITM_NAME), '') <> ''
    """).assign(table=lambda d: d.file.map(data.table))
    described = described.merge(dictionary.drop_duplicates(["table", "column"])[["table", "column", "description"]], how="left", on=["table", "column"])
described.groupby(["layer", "table"]).column.apply(", ".join).rename("columns").reset_index()

To read one table's columns with their types and descriptions, filter `described`, for example `described.query("table == 'mds_updrs_part_iii'")`.